# Práctica Machine Learning — Predicción de precios de Airbnb

## Descripción del problema

El objetivo de esta práctica es abordar un problema de **regresión** sobre datos reales: predecir el **precio por noche** de un alojamiento de Airbnb a partir de sus características (ubicación, tipo de alojamiento, capacidad, reseñas, etc.).

El enfoque es **incremental**: partimos de modelos simples para ir aumentando la complejidad, comparando resultados en cada paso.

## Metodología

```
1. Carga y división train/test
   └─ Separar ANTES de explorar para evitar data leakage

2. Análisis Exploratorio (EDA)
   ├─ Valores nulos
   ├─ Distribución del target
   ├─ Detección de outliers (IQR)
   ├─ Correlación con el precio
   ├─ Variables categóricas
   └─ Identificación de columnas a eliminar

3. Preprocesamiento
   ├─ Eliminación de columnas (URLs, texto, leakage, alta cardinalidad)
   ├─ Eliminación de outliers en train
   ├─ Imputación de nulos (mediana / "Unknown")
   ├─ Simplificación de categóricas + One-Hot Encoding
   └─ Selección de variables (importancia con Random Forest)

4. Modelado (Cross-Validation K=5)
   ├─ Regresión Lineal
   ├─ Ridge
   ├─ Random Forest
   └─ Gradient Boosting

5. Evaluación final en Test
   ├─ Métricas: MAE, RMSE, R²
   ├─ Análisis de errores del mejor modelo
   └─ Importancia de variables

6. Conclusiones
```

## 1. Carga de datos y división train/test

Dividimos antes de explorar para evitar data leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('airbnb-listings-extract.csv', sep=';')
df = df.dropna(subset=['Price'])

print(f'Dataset: {df.shape[0]} filas × {df.shape[1]} columnas')

X = df.drop(columns=['Price'])
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} ({X_train.shape[0]/len(df)*100:.0f}%)')
print(f'Test:  {X_test.shape[0]} ({X_test.shape[0]/len(df)*100:.0f}%)')

## 2. Análisis Exploratorio (EDA)

Todo el análisis se realiza sobre train para no contaminar el test.

In [ ]:
train = X_train.copy()
train['Price'] = y_train

# Tipos de datos
print(f'Numéricas: {train.select_dtypes(include="number").shape[1]}')
print(f'Categóricas: {train.select_dtypes(include="object").shape[1]}')

### 2.1 Valores nulos

In [ ]:
# Columnas con nulos, ordenadas por porcentaje
nulls = (train.isna().mean() * 100).sort_values(ascending=False)
nulls = nulls[nulls > 0]

print(f'Columnas con algún nulo: {len(nulls)} de {train.shape[1]}')
for col, pct in nulls.items():
    print(f'  {col:40s} {pct:5.1f}%')

### 2.2 Distribución del precio

In [ ]:
print(y_train.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y_train, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(y_train.mean(), color='red', linestyle='--', label=f'Media: {y_train.mean():.0f}$')
axes[0].axvline(y_train.median(), color='orange', linestyle='--', label=f'Mediana: {y_train.median():.0f}$')
axes[0].set_title('Distribución del precio')
axes[0].set_xlabel('Precio ($/noche)')
axes[0].legend()

axes[1].hist(y_train[y_train < 500], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Zoom: precio < 500$')
axes[1].set_xlabel('Precio ($/noche)')

plt.tight_layout()
plt.show()

# Distribución sesgada a la derecha: media (74$) > mediana (55$)

### 2.3 Detección de outliers (método IQR)

In [ ]:
Q1 = y_train.quantile(0.25)
Q3 = y_train.quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

n_outliers = (y_train > limite_superior).sum()

print(f'IQR: {IQR:.0f}$, Límite superior: {limite_superior:.0f}$')
print(f'Outliers: {n_outliers} pisos ({n_outliers/len(y_train)*100:.1f}%) con precio > {limite_superior:.0f}$')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].boxplot(y_train, vert=False)
axes[0].set_title('Con outliers')
axes[1].boxplot(y_train[y_train <= limite_superior], vert=False)
axes[1].set_title(f'Sin outliers (≤{limite_superior:.0f}$)')
plt.tight_layout()
plt.show()

### 2.4 Correlación con el precio

In [ ]:
num_cols = ['Accommodates', 'Bathrooms', 'Bedrooms', 'Beds', 'Guests Included',
            'Extra People', 'Minimum Nights', 'Maximum Nights',
            'Number of Reviews', 'Review Scores Rating', 'Reviews per Month',
            'Host Listings Count', 'Cleaning Fee', 'Security Deposit',
            'Availability 30', 'Availability 365', 'Latitude', 'Longitude',
            'Calculated host listings count', 'Price']

correlaciones = train[num_cols].corr()['Price'].drop('Price').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['green' if abs(x) > 0.3 else 'orange' if abs(x) > 0.15 else 'lightgrey' for x in correlaciones]
correlaciones.plot(kind='barh', color=colors, ax=ax)
ax.set_title('Correlación con el precio')
ax.axvline(x=0.3, color='green', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(x=-0.3, color='green', linewidth=0.8, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Las variables con mayor correlación son Cleaning Fee (0.67), Accommodates (0.52),
# Bedrooms (0.50), Beds (0.44) y Bathrooms (0.39)

### 2.5 Variables categóricas: precio medio por categoría

In [ ]:
cat_cols = ['Room Type', 'Property Type', 'Cancellation Policy', 'Bed Type']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    medias = train.groupby(col)['Price'].mean().sort_values(ascending=True)
    if len(medias) > 10:
        medias = medias.tail(10)
    medias.plot(kind='barh', ax=axes[i], color='steelblue')
    axes[i].set_title(f'Precio medio por {col}')
    axes[i].set_xlabel('Precio medio ($)')

plt.tight_layout()
plt.show()

# Room Type es la categórica más relevante:
# Entire home (96$) vs Private room (41$) vs Shared room (31$)

### 2.6 Identificación de columnas a eliminar

| Razón | Columnas | 
|-------|----------|
| URLs / IDs | Listing Url, Host ID, Picture Url, Geolocation... |
| Texto libre | Description, Summary, House Rules, Amenities... |
| >70% nulos | Square Feet (96%), License (98%), Host Acceptance Rate (99.7%) |
| Casi constantes | Experiences Offered (99.9% "none") |
| Data leakage | Weekly Price, Monthly Price (derivados del precio diario) |

## 3. Preprocesamiento

### 3.1 Eliminación de columnas

In [ ]:
cols_urls_ids = ['ID', 'Listing Url', 'Scrape ID', 'Last Scraped',
                 'Thumbnail Url', 'Medium Url', 'Picture Url', 'XL Picture Url',
                 'Host URL', 'Host Thumbnail Url', 'Host Picture Url',
                 'Host ID', 'Calendar last Scraped', 'Geolocation']

cols_texto = ['Name', 'Summary', 'Space', 'Description', 'Experiences Offered',
              'Neighborhood Overview', 'Notes', 'Transit', 'Access',
              'Interaction', 'House Rules', 'Host About', 'Host Name',
              'Host Location', 'Street', 'Smart Location', 'Host Since',
              'Host Verifications', 'Features', 'Amenities',
              'First Review', 'Last Review', 'Calendar Updated']

cols_nulos = ['Square Feet', 'Has Availability', 'License',
              'Jurisdiction Names', 'Host Acceptance Rate']

cols_leakage = ['Weekly Price', 'Monthly Price']
cols_redundantes = ['Country', 'Country Code']

# Categóricas con cardinalidad excesiva
cols_alta_card = ['City', 'State', 'Zipcode', 'Market', 'Neighbourhood',
                  'Neighbourhood Group Cleansed', 'Host Neighbourhood']

cols_eliminar = (cols_urls_ids + cols_texto + cols_nulos + 
                 cols_leakage + cols_redundantes + cols_alta_card)

X_train_clean = X_train.drop(columns=cols_eliminar, errors='ignore')
X_test_clean = X_test.drop(columns=cols_eliminar, errors='ignore')

print(f'{X_train.shape[1]} → {X_train_clean.shape[1]} columnas')

### 3.2 Eliminación de outliers en train

Eliminamos del train los pisos por encima del límite IQR. No tocamos el test.

In [ ]:
mascara = y_train <= limite_superior
X_train_clean = X_train_clean[mascara]
y_train_clean = y_train[mascara]

print(f'Train: {len(y_train)} → {len(y_train_clean)} pisos (eliminados {len(y_train)-len(y_train_clean)} con precio > {limite_superior:.0f}$)')

### 3.3 Imputación de nulos

Numéricas con la mediana del train, categóricas con "Unknown". Usamos los valores del train también para el test.

In [ ]:
# Numéricas: mediana del train
for col in X_train_clean.select_dtypes(include='number').columns:
    if X_train_clean[col].isna().any():
        med = X_train_clean[col].median()
        X_train_clean[col] = X_train_clean[col].fillna(med)
        X_test_clean[col] = X_test_clean[col].fillna(med)

# Categóricas: "Unknown"
for col in X_train_clean.select_dtypes(include='object').columns:
    if X_train_clean[col].isna().any():
        X_train_clean[col] = X_train_clean[col].fillna('Unknown')
        X_test_clean[col] = X_test_clean[col].fillna('Unknown')

print(f'Nulos en train: {X_train_clean.isna().sum().sum()}')
print(f'Nulos en test:  {X_test_clean.isna().sum().sum()}')

### 3.4 Simplificación de categóricas y One-Hot Encoding

Agrupamos en "Other" las categorías con menos de 50 ejemplos y aplicamos One-Hot Encoding.

In [ ]:
# Agrupar categorías minoritarias
def simplificar(train_col, test_col, min_count=50):
    counts = train_col.value_counts()
    valid = counts[counts >= min_count].index
    return (train_col.where(train_col.isin(valid), 'Other'),
            test_col.where(test_col.isin(valid), 'Other'))

for col in X_train_clean.select_dtypes(include='object').columns:
    X_train_clean[col], X_test_clean[col] = simplificar(X_train_clean[col], X_test_clean[col])

# One-Hot Encoding
X_train_enc = pd.get_dummies(X_train_clean, drop_first=True)
X_test_enc = pd.get_dummies(X_test_clean, drop_first=True)

# Alinear columnas entre train y test
cols_comunes = X_train_enc.columns.intersection(X_test_enc.columns)
X_train_enc = X_train_enc[cols_comunes]
X_test_enc = X_test_enc[cols_comunes]

print(f'Dimensiones finales:')
print(f'  Train: {X_train_enc.shape[0]} × {X_train_enc.shape[1]}')
print(f'  Test:  {X_test_enc.shape[0]} × {X_test_enc.shape[1]}')

### 3.5 Importancia de variables (Random Forest)

Utilizamos un Random Forest preliminar para identificar las variables más relevantes.

In [ ]:
rf_prelim = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_prelim.fit(X_train_enc, y_train_clean)

importancias = pd.Series(rf_prelim.feature_importances_, index=X_train_enc.columns)
importancias = importancias.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
importancias.head(15).plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Top 15 variables más importantes')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Room Type es con diferencia la variable más determinante, seguida de
# Bathrooms, ubicación (Lat/Lon), Cleaning Fee y Bedrooms.

## 4. Modelado con Cross-Validation (K=5)

Probamos 4 modelos de complejidad creciente y comparamos su MAE en cross-validation.

In [ ]:
modelos = {
    'Regresión Lineal': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
}

resultados_cv = {}

for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train_enc, y_train_clean, cv=5, scoring='neg_mean_absolute_error')
    mae = -scores.mean()
    std = scores.std()
    resultados_cv[nombre] = {'MAE': mae, 'Std': std}
    print(f'{nombre:25s} | MAE: {mae:.2f}$ (±{std:.2f}$)')

mejor_cv = min(resultados_cv, key=lambda k: resultados_cv[k]['MAE'])
print(f'\nMejor modelo en CV: {mejor_cv} ({resultados_cv[mejor_cv]["MAE"]:.2f}$)')

In [ ]:
# Comparativa visual
df_cv = pd.DataFrame(resultados_cv).T.sort_values('MAE')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['gold' if x == df_cv['MAE'].min() else 'steelblue' for x in df_cv['MAE']]
bars = ax.barh(df_cv.index, df_cv['MAE'], color=colors)
ax.set_xlabel('MAE ($)')
ax.set_title('Cross-Validation: MAE por modelo')
for bar, val in zip(bars, df_cv['MAE']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}$', va='center')
plt.tight_layout()
plt.show()

## 5. Evaluación final en Test

Entrenamos cada modelo con todo el train y evaluamos en test (datos no vistos durante el entrenamiento).

In [ ]:
resultados_test = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train_enc, y_train_clean)
    y_pred = modelo.predict(X_test_enc)
    
    resultados_test.append({
        'Modelo': nombre,
        'MAE ($)': round(mean_absolute_error(y_test, y_pred), 2),
        'RMSE ($)': round(np.sqrt(mean_squared_error(y_test, y_pred)), 2),
        'R²': round(r2_score(y_test, y_pred), 3)
    })

df_test = pd.DataFrame(resultados_test).sort_values('MAE ($)')
print(df_test.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_mae = ['gold' if x == df_test['MAE ($)'].min() else 'steelblue' for x in df_test['MAE ($)']]
bars1 = axes[0].barh(df_test['Modelo'], df_test['MAE ($)'], color=colors_mae)
axes[0].set_xlabel('MAE ($)')
axes[0].set_title('MAE en Test (menor = mejor)')
for bar, val in zip(bars1, df_test['MAE ($)']):
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, f'{val}$', va='center')

colors_r2 = ['gold' if x == df_test['R²'].max() else 'steelblue' for x in df_test['R²']]
bars2 = axes[1].barh(df_test['Modelo'], df_test['R²'], color=colors_r2)
axes[1].set_xlabel('R²')
axes[1].set_title('R² en Test (mayor = mejor)')
axes[1].set_xlim(0, 1)
for bar, val in zip(bars2, df_test['R²']):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{val}', va='center')

plt.tight_layout()
plt.show()

### 5.1 Análisis del mejor modelo

In [ ]:
# Predicciones del Gradient Boosting
mejor = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
mejor.fit(X_train_enc, y_train_clean)
y_pred_mejor = mejor.predict(X_test_enc)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Predicho vs Real
axes[0].scatter(y_test, y_pred_mejor, alpha=0.3, s=10, color='steelblue')
axes[0].plot([0, 500], [0, 500], 'r--', label='Predicción perfecta')
axes[0].set_xlabel('Precio real ($)')
axes[0].set_ylabel('Precio predicho ($)')
axes[0].set_title('Predicho vs Real')
axes[0].set_xlim(0, 500)
axes[0].set_ylim(0, 300)
axes[0].legend()

# Distribución de errores
errores = y_test - y_pred_mejor
axes[1].hist(errores, bins=50, color='coral', edgecolor='white')
axes[1].axvline(x=0, color='black', linestyle='--')
axes[1].set_xlabel('Error (real - predicho, $)')
axes[1].set_title('Distribución de errores')

plt.tight_layout()
plt.show()

print(f'El {(abs(errores) < 20).mean()*100:.1f}% de pisos se predice con error < 20$')
print(f'El {(abs(errores) > 50).mean()*100:.1f}% tiene error > 50$ (principalmente pisos muy caros)')

In [ ]:
# Variables más importantes del modelo final
imp = pd.Series(mejor.feature_importances_, index=X_train_enc.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
imp.head(15).plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Variables más importantes (Gradient Boosting)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Conclusiones

Hemos abordado un problema de regresión sobre datos reales de Airbnb, siguiendo un proceso incremental: exploración, preprocesamiento, y modelado con cuatro algoritmos de complejidad creciente.

El mejor modelo fue **Gradient Boosting** (MAE ≈ 20$, R² ≈ 0.39). Las variables más determinantes del precio son el tipo de alojamiento (piso entero vs. habitación), el número de baños y habitaciones, la ubicación y la tarifa de limpieza.

El R² no es alto: el precio de un Airbnb depende de muchos factores que no están en el dataset como fotos, decoración, temporada, reputación del anfitrión y demás. El modelo tiene especial dificultad con pisos muy caros, que se eliminaron como outliers durante el entrenamiento.